In [1]:
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools
import logging
from one.api import ONE
from brainbox.io.one import SessionLoader
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from collections import defaultdict
import pandas as pd
from manifold.decoding.functions.utils import check_config_decoding
import numpy as np
import pickle as pkl
from manifold.decoding.functions import nulldistributions
from communication_subspace.ibl_communication.utils import load_widefield_epoch
from tqdm import tqdm

from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, balanced_accuracy_score
import warnings
from matplotlib import pyplot as plt
warnings.filterwarnings("ignore")
from manifold.utils import get_trial_masks
from glob import glob
import numpy as np
import pandas as pd
from scipy.stats import ttest_1samp
import statsmodels.stats.multitest as smm
import seaborn as sns
import pickle as pkl
from glob import glob
from manifold.widefield_ppi import return_labels

In [2]:
%load_ext autoreload 
%autoreload 2

In [3]:
from prior_localization.fit_data import fit_session_widefield

In [4]:
from pathlib import Path


one = ONE(mode="local")

significant_pkl_path = Path("../data/processed/significant_stims_choice.pkl")

with open(significant_pkl_path, "rb") as f:
    significant_pickles = pkl.load(f)


In [5]:
eid = list(significant_pickles.keys())[0]
print("Running on", eid)

subject = one.get_details(eid)['subject']
print("Subject:", subject)


Running on f7d46a15-9498-40dc-90da-fb977ce844be
Subject: CSK-im-009


In [6]:
n_pseudo=2

In [7]:
all_pseudo = list(range(n_pseudo))
# Select relevant pseudo sessions for this job
pseudo_ids = all_pseudo
pseudo_ids = list(np.array(pseudo_ids) + 1)
pseudo_ids = [-1] + pseudo_ids

In [8]:
from prior_localization.functions.utils import check_config

In [9]:
cf = check_config()

In [16]:
x = fit_session_widefield(
            one=one,
            session_id=eid,
            subject=subject,
            output_dir=Path("../data/generated/prior_sig"),
            pseudo_ids=pseudo_ids,
            hemisphere=("left", "right"),
            target="prior",
            align_event="stimOn_times",
            frame_window=(-2, -2),
            model="actKernel",
            n_runs=1,
        )

what even is [['VISp'], ['MOs']]
2026-08-24 16:14:23 INFO     base_models.py:411  results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:23,502 [INFO] results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:23 INFO     base_models.py:411  results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:23,717 [INFO] results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:23 INFO     base_models.py:411  results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


...

2026-08-24 16:14:24,480 [INFO] results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:24 INFO     base_models.py:411  results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:24,694 [INFO] results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:24 INFO     base_models.py:411  results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


2026-08-24 16:14:24,822 [INFO] results found and loaded from ../data/generated/prior_sig/behavior/CSK-im-009/model_actKernel_single_zeta/train_f7d46a15.pkl


In [84]:
data = pkl.load(open('../data/generated/prior_sig/CSK-im-009/f7d46a15-9498-40dc-90da-fb977ce844be/VISp_both_hemispheres_pseudo_ids_-1_10.pkl','rb'))

In [86]:
data.keys()

dict_keys(['fit', 'subject', 'eid', 'hemisphere', 'region', 'N_units'])

In [15]:
x

[PosixPath('../data/generated/prior_sig/CSK-im-009/f7d46a15-9498-40dc-90da-fb977ce844be/VISp_MOs_both_hemispheres_pseudo_ids_-1_2.pkl'),
 PosixPath('../data/generated/prior_sig/CSK-im-009/f7d46a15-9498-40dc-90da-fb977ce844be/VISp_MOs_both_hemispheres_pseudo_ids_-1_2.pkl')]